<a href="https://colab.research.google.com/github/SalmaAnindhia/PemMes_25_Salma/blob/main/JS09/JS09_Tugas1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q annoy faiss-cpu hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import numpy as np
import pandas as pd
import time
import gc
import faiss
import hnswlib
from annoy import AnnoyIndex

In [3]:
k = 10
nq = 100

def make_data(n, dim, nq=nq, seed=42):
    rng = np.random.default_rng(seed)
    X = rng.random((n, dim), dtype=np.float32)
    Q = rng.random((nq, dim), dtype=np.float32)
    return X, Q

def recall_at_k(I_pred, I_gt, k):
    correct = 0
    for i in range(len(I_pred)):
        correct += len(set(I_pred[i][:k]) & set(I_gt[i][:k]))
    return correct / (len(I_pred) * k)

In [4]:
def run_exact(X, Q, k, metric):
    d = X.shape[1]
    if metric == 'euclidean':
        index = faiss.IndexFlatL2(d)
        Xs, Qs = X, Q
    else:
        Xs, Qs = X.copy(), Q.copy()
        faiss.normalize_L2(Xs); faiss.normalize_L2(Qs)
        index = faiss.IndexFlatIP(d)

    t0 = time.time()
    index.add(Xs)
    build = time.time() - t0

    faiss.omp_set_num_threads(1)
    t0 = time.time()
    _, I = index.search(Qs, k)
    qms = (time.time() - t0) * 1000 / len(Qs)
    faiss.omp_set_num_threads(faiss.omp_get_max_threads())
    return I, build, qms

In [5]:
def run_annoy(X, Q, k, metric, n_trees=10):
    d = X.shape[1]
    index = AnnoyIndex(d, 'euclidean' if metric == 'euclidean' else 'angular')

    t0 = time.time()
    for i, v in enumerate(X):
        index.add_item(i, v)
    index.build(n_trees)
    build = time.time() - t0

    t0 = time.time()
    I = [index.get_nns_by_vector(q, k) for q in Q]
    qms = (time.time() - t0) * 1000 / len(Q)
    return I, build, qms

In [6]:
def run_faiss(X, Q, k, metric, nprobe=10):
    n, d = X.shape
    nlist = max(1, min(int(np.sqrt(n)), n // 39))
    nprobe = min(nprobe, nlist)

    if metric == 'euclidean':
        Xs, Qs = X, Q
        quantizer = faiss.IndexFlatL2(d)
        index = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_L2)
    else:
        Xs, Qs = X.copy(), Q.copy()
        faiss.normalize_L2(Xs); faiss.normalize_L2(Qs)
        quantizer = faiss.IndexFlatIP(d)
        index = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)

    t0 = time.time()
    index.train(Xs[:min(n, 100_000)])
    index.add(Xs)
    build = time.time() - t0

    index.nprobe = nprobe
    faiss.omp_set_num_threads(1)
    t0 = time.time()
    _, I = index.search(Qs, k)
    qms = (time.time() - t0) * 1000 / len(Qs)
    faiss.omp_set_num_threads(faiss.omp_get_max_threads())
    return I, build, qms

In [7]:
def run_hnsw(X, Q, k, metric, M=16, ef_construction=100, ef=50):
    n, d = X.shape
    p = hnswlib.Index(space='l2' if metric == 'euclidean' else 'cosine', dim=d)

    t0 = time.time()
    p.init_index(max_elements=n, ef_construction=ef_construction, M=M)
    p.add_items(X)
    build = time.time() - t0

    p.set_ef(ef)
    p.set_num_threads(1)
    t0 = time.time()
    I, _ = p.knn_query(Q, k)
    qms = (time.time() - t0) * 1000 / len(Q)
    return I, build, qms

In [8]:
rows = []

for n in [1000, 1_000_000]:
    for dim in [2, 5]:
        X, Q = make_data(n, dim)
        for metric in ['euclidean', 'cosine']:
            print(f"Running: n={n}, dim={dim}D, metric={metric}")

            I_gt, b, q = run_exact(X, Q, k, metric)
            rows.append(dict(Data=f"{n}/{dim}D", Metric=metric, Algo="Exact NN",
                             Build_s=b, Query_ms=q, Recall=1.0))

            for name, fn in [("ANNOY", run_annoy), ("FAISS", run_faiss), ("HNSW", run_hnsw)]:
                I, b, q = fn(X, Q, k, metric)
                rows.append(dict(Data=f"{n}/{dim}D", Metric=metric, Algo=name,
                                 Build_s=b, Query_ms=q, Recall=recall_at_k(I, I_gt, k)))
                del I; gc.collect()
        del X, Q; gc.collect()

df_res = pd.DataFrame(rows)
df_res

Running: n=1000, dim=2D, metric=euclidean
Running: n=1000, dim=2D, metric=cosine
Running: n=1000, dim=5D, metric=euclidean
Running: n=1000, dim=5D, metric=cosine
Running: n=1000000, dim=2D, metric=euclidean
Running: n=1000000, dim=2D, metric=cosine
Running: n=1000000, dim=5D, metric=euclidean
Running: n=1000000, dim=5D, metric=cosine


,Data,Metric,Algo,Build_s,Query_ms,Recall
0,1000/2D,euclidean,Exact NN,0.000100,0.008471,1.000
1,1000/2D,euclidean,ANNOY,0.016265,0.019519,0.999
2,1000/2D,euclidean,FAISS,0.001842,0.004764,1.000
3,1000/2D,euclidean,HNSW,0.017073,0.008733,1.000
4,1000/2D,cosine,Exact NN,0.000022,0.008249,1.000
5,1000/2D,cosine,ANNOY,0.016709,0.016291,0.997
6,1000/2D,cosine,FAISS,0.001931,0.005481,1.000
7,1000/2D,cosine,HNSW,0.012044,0.007730,1.000
8,1000/5D,euclidean,Exact NN,0.000711,0.006723,1.000
9,1000/5D,euclidean,ANNOY,0.009341,0.016849,0.968
